# Chicago 311: Do Wealthier Neighborhoods Get Faster City Services?

**An equity analysis of 3 years of Chicago 311 service requests**

Chicago receives millions of 311 service requests each year — potholes, streetlights, rodent complaints, graffiti, and more. This analysis investigates whether response times and service quality vary systematically by neighborhood income level.

**Research Question:** Do wealthier Chicago neighborhoods receive faster 311 service responses than lower-income neighborhoods?

**Approach:**
1. Pull 3 years of completed 311 requests with location data
2. Join with Census socioeconomic data by community area
3. Compare response times across income quartiles
4. Use NLP to understand what different neighborhoods report
5. Test significance with statistical methods

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.options.display.float_format = '{:.2f}'.format

DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

## 1. Load Data

We use two datasets:
- **311 Service Requests** — completed requests from 2023–2026 with location, timestamps, and service type (via Chicago Data Portal)
- **Community Area Socioeconomic Indicators** — per-capita income, poverty rate, and hardship index for Chicago's 77 community areas (Census ACS via Chicago Data Portal)

In [ ]:
# Load 311 service requests
df = pd.read_parquet(DATA_DIR / "chicago_311.parquet")
print(f"311 requests: {len(df):,} rows")
print(f"Date range: {df['created_date'].min():%Y-%m-%d} to {df['created_date'].max():%Y-%m-%d}")
print(f"Service types: {df['sr_type'].nunique()}")
print(f"Community areas: {df['community_area'].dropna().nunique()}")
df.head(3)

In [ ]:
# Load community area socioeconomic data
socio = pd.read_parquet(DATA_DIR / "community_area_socioeconomic.parquet")
print(f"Community areas: {len(socio)}")
socio.head(3)

In [ ]:
# Inspect socioeconomic columns to find income/hardship fields
print("Socioeconomic columns:")
for col in socio.columns:
    print(f"  {col}: {socio[col].dtype} — e.g. {socio[col].iloc[0]}")


In [ ]:
# Join 311 data with socioeconomic indicators by community area
# First, identify the join key and income field from the socio data
# (we'll adapt column names after inspecting the data above)

# Standardize community area as int for join
df["community_area"] = df["community_area"].astype("Int64")

# Attempt join — column names will be confirmed after first run
socio_cols = socio.columns.tolist()
print("Available socio columns:", socio_cols)

# Look for community area number and per-capita income
ca_col = [c for c in socio_cols if "community" in c.lower() and "number" in c.lower() or "area" in c.lower() and "num" in c.lower()]
income_col = [c for c in socio_cols if "capita" in c.lower() or "income" in c.lower()]
hardship_col = [c for c in socio_cols if "hardship" in c.lower()]
print(f"CA column candidates: {ca_col}")
print(f"Income column candidates: {income_col}")
print(f"Hardship column candidates: {hardship_col}")

## 2. Exploratory Data Analysis

Before testing our equity hypothesis, let's understand the shape of the data — what types of requests are most common, how response times are distributed, and how requests vary across the city.

In [ ]:
# Top 15 service request types
top_types = df["sr_type"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_types.plot.barh(ax=ax, color="#4a90d9", edgecolor="white")
ax.set_xlabel("Number of Requests")
ax.set_title("Top 15 Service Request Types (2023–2026)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "top_request_types.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Response time distribution (cap at 30 days for visualization)
response_days = df["response_hours"] / 24
print(f"Median response time: {response_days.median():.1f} days")
print(f"Mean response time: {response_days.mean():.1f} days")
print(f"90th percentile: {response_days.quantile(0.9):.1f} days")
print(f"Requests resolved same day: {(response_days < 1).mean():.1%}")

fig, ax = plt.subplots(figsize=(10, 5))
response_days.clip(upper=30).hist(bins=60, ax=ax, color="#4a90d9", edgecolor="white", alpha=0.8)
ax.set_xlabel("Days to Resolution")
ax.set_ylabel("Number of Requests")
ax.set_title("Response Time Distribution (capped at 30 days)")
ax.axvline(response_days.median(), color="#e74c3c", linestyle="--", linewidth=2, label=f"Median: {response_days.median():.1f} days")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "response_time_dist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Response time by request type (top 10)
top10 = df[df["sr_type"].isin(top_types.head(10).index)].copy()
top10["response_days"] = top10["response_hours"] / 24

medians = top10.groupby("sr_type")["response_days"].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
medians.plot.barh(ax=ax, color="#4a90d9", edgecolor="white")
ax.set_xlabel("Median Days to Resolution")
ax.set_title("Median Response Time by Service Type")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "response_by_type.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Equity Analysis: Response Time by Income

This is the core question. We join 311 requests with community area income data, divide neighborhoods into income quartiles, and compare response times.

If the city delivers services equitably, we should see **no significant difference** in response times across income groups.

In [ ]:
# Join 311 data with community area income
# Adapt column names based on what we found in the socio data inspection (cell 5/6)
# The Chicago Data Portal socioeconomic dataset typically has:
#   community_area_number, community_area_name, per_capita_income_, hardship_index

# Normalize column names
socio.columns = socio.columns.str.lower().str.strip().str.replace(" ", "_")
print("Normalized socio columns:", socio.columns.tolist())

# Find the right columns
ca_num_col = [c for c in socio.columns if "community_area" in c and ("number" in c or "num" in c)]
income_cols = [c for c in socio.columns if "capita" in c or "income" in c]
hardship_cols = [c for c in socio.columns if "hardship" in c]
poverty_cols = [c for c in socio.columns if "poverty" in c]

print(f"\nCommunity area number col: {ca_num_col}")
print(f"Income cols: {income_cols}")
print(f"Hardship cols: {hardship_cols}")
print(f"Poverty cols: {poverty_cols}")

# Build a clean lookup
ca_col = ca_num_col[0] if ca_num_col else socio.columns[0]
inc_col = income_cols[0] if income_cols else None

socio[ca_col] = pd.to_numeric(socio[ca_col], errors="coerce").astype("Int64")
if inc_col:
    socio[inc_col] = pd.to_numeric(socio[inc_col], errors="coerce")

# Merge
df = df.merge(
    socio[[ca_col, inc_col] + hardship_cols + poverty_cols].rename(columns={ca_col: "community_area"}),
    on="community_area",
    how="left",
)

# Rename income column for clarity
if inc_col and inc_col != "per_capita_income":
    df = df.rename(columns={inc_col: "per_capita_income"})
    inc_col = "per_capita_income"

matched = df["per_capita_income"].notna().sum()
print(f"\nMatched {matched:,} / {len(df):,} requests ({matched/len(df):.1%}) with income data")

In [ ]:
# Create income quartiles
df_inc = df.dropna(subset=["per_capita_income", "response_hours"]).copy()
df_inc["response_days"] = df_inc["response_hours"] / 24

# Assign quartile based on community area income
income_quartiles = df_inc.groupby("community_area")["per_capita_income"].first()
q_labels = ["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"]
quartile_bins = pd.qcut(income_quartiles, 4, labels=q_labels)
ca_to_quartile = quartile_bins.to_dict()
df_inc["income_quartile"] = df_inc["community_area"].map(ca_to_quartile)

# Summary stats by quartile
summary = df_inc.groupby("income_quartile").agg(
    requests=("sr_number", "count"),
    median_days=("response_days", "median"),
    mean_days=("response_days", "mean"),
    pct_same_day=("response_days", lambda x: (x < 1).mean() * 100),
    avg_income=("per_capita_income", "mean"),
).round(2)

print("Response Time by Income Quartile")
print("=" * 70)
summary

In [ ]:
# Visualize response time by income quartile
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot (capped at 30 days for readability)
colors = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]
data_by_q = [df_inc[df_inc["income_quartile"] == q]["response_days"].clip(upper=30) for q in q_labels]
bp = axes[0].boxplot(data_by_q, labels=q_labels, patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_ylabel("Days to Resolution")
axes[0].set_title("Response Time Distribution by Income Quartile")

# Bar chart of medians
axes[1].bar(q_labels, summary["median_days"], color=colors, edgecolor="white", alpha=0.8)
axes[1].set_ylabel("Median Days to Resolution")
axes[1].set_title("Median Response Time by Income Quartile")

plt.tight_layout()
plt.savefig(FIG_DIR / "response_by_income.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Statistical Testing

Visual differences could be due to chance. We use two statistical tests:

1. **Kruskal-Wallis H-test** — non-parametric test for whether response time distributions differ across income quartiles (appropriate because response times are heavily right-skewed)
2. **Mann-Whitney U test** — pairwise comparison between the lowest and highest income quartiles
3. **Bootstrap confidence intervals** — estimate the true difference in medians

In [ ]:
# Kruskal-Wallis test across all 4 income quartiles
groups = [g["response_days"].values for _, g in df_inc.groupby("income_quartile")]
h_stat, p_kw = stats.kruskal(*groups)
print("Kruskal-Wallis H-test (all 4 quartiles)")
print(f"  H-statistic: {h_stat:.1f}")
print(f"  p-value: {p_kw:.2e}")
print(f"  Significant at α=0.05? {'Yes' if p_kw < 0.05 else 'No'}")

# Mann-Whitney U: Q1 (lowest income) vs Q4 (highest income)
q1 = df_inc[df_inc["income_quartile"] == "Q1 (Lowest)"]["response_days"]
q4 = df_inc[df_inc["income_quartile"] == "Q4 (Highest)"]["response_days"]
u_stat, p_mw = stats.mannwhitneyu(q1, q4, alternative="two-sided")
print(f"\nMann-Whitney U test: Q1 (Lowest) vs Q4 (Highest)")
print(f"  U-statistic: {u_stat:,.0f}")
print(f"  p-value: {p_mw:.2e}")
print(f"  Q1 median: {q1.median():.2f} days")
print(f"  Q4 median: {q4.median():.2f} days")
print(f"  Difference: {q1.median() - q4.median():.2f} days")

In [ ]:
# Bootstrap confidence interval for the difference in medians (Q1 - Q4)
np.random.seed(42)
n_boot = 10000
boot_diffs = []

for _ in range(n_boot):
    q1_sample = q1.sample(n=min(5000, len(q1)), replace=True)
    q4_sample = q4.sample(n=min(5000, len(q4)), replace=True)
    boot_diffs.append(q1_sample.median() - q4_sample.median())

boot_diffs = np.array(boot_diffs)
ci_lower = np.percentile(boot_diffs, 2.5)
ci_upper = np.percentile(boot_diffs, 97.5)

print(f"Bootstrap 95% CI for median difference (Q1 - Q4):")
print(f"  [{ci_lower:.2f}, {ci_upper:.2f}] days")
print(f"  Point estimate: {np.median(boot_diffs):.2f} days")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_diffs, bins=50, color="#4a90d9", edgecolor="white", alpha=0.8)
ax.axvline(0, color="black", linestyle="-", linewidth=1.5, label="No difference")
ax.axvline(ci_lower, color="#e74c3c", linestyle="--", linewidth=1.5, label=f"95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")
ax.axvline(ci_upper, color="#e74c3c", linestyle="--", linewidth=1.5)
ax.set_xlabel("Difference in Median Response (Q1 − Q4, days)")
ax.set_ylabel("Bootstrap Samples")
ax.set_title("Bootstrap Distribution: Response Time Gap (Low vs High Income)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "bootstrap_ci.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. What Do Different Neighborhoods Report?

Beyond response times, we can examine *what* different income groups report. Do lower-income neighborhoods report more infrastructure issues (potholes, streetlights) while higher-income areas report quality-of-life concerns (graffiti, trees)?

This tells a richer story about how different communities experience city services.

In [ ]:
# Service type mix by income quartile
# Focus on top 10 types for readability
top10_types = df_inc["sr_type"].value_counts().head(10).index

type_by_q = (
    df_inc[df_inc["sr_type"].isin(top10_types)]
    .groupby(["income_quartile", "sr_type"])
    .size()
    .unstack(fill_value=0)
)

# Normalize to percentages within each quartile
type_pct = type_by_q.div(type_by_q.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 6))
type_pct.plot.bar(stacked=True, ax=ax, colormap="tab10", edgecolor="white", linewidth=0.5)
ax.set_ylabel("% of Requests")
ax.set_xlabel("")
ax.set_title("Service Request Mix by Income Quartile")
ax.legend(title="Request Type", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "request_mix_by_income.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Heatmap: median response time by request type × income quartile
pivot = (
    df_inc[df_inc["sr_type"].isin(top10_types)]
    .groupby(["income_quartile", "sr_type"])["response_days"]
    .median()
    .unstack()
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r", ax=ax, linewidths=0.5)
ax.set_title("Median Response Time (Days) by Request Type × Income Quartile")
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_type_income.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Controlling for Request Type

A key confound: if lower-income neighborhoods report more complex issues (e.g., building violations vs. garbage carts), slower response times could reflect issue complexity rather than inequity.

We control for this by comparing response times **within the same request type** across income quartiles.

In [ ]:
# Within-type comparison: for each of the top 10 types,
# test whether Q1 and Q4 have different response times
print(f"{'Request Type':<40} {'Q1 Median':>10} {'Q4 Median':>10} {'Diff':>8} {'p-value':>10}")
print("=" * 85)

within_type_results = []
for sr_type in top10_types:
    subset = df_inc[df_inc["sr_type"] == sr_type]
    q1_sub = subset[subset["income_quartile"] == "Q1 (Lowest)"]["response_days"]
    q4_sub = subset[subset["income_quartile"] == "Q4 (Highest)"]["response_days"]

    if len(q1_sub) < 30 or len(q4_sub) < 30:
        continue

    _, p = stats.mannwhitneyu(q1_sub, q4_sub, alternative="two-sided")
    diff = q1_sub.median() - q4_sub.median()
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

    within_type_results.append({
        "sr_type": sr_type, "q1_median": q1_sub.median(), "q4_median": q4_sub.median(),
        "diff": diff, "p_value": p, "sig": sig,
    })
    print(f"{sr_type:<40} {q1_sub.median():>10.2f} {q4_sub.median():>10.2f} {diff:>+8.2f} {p:>10.2e} {sig}")

print(f"\n{'*'} p<0.05  {'**'} p<0.01  {'***'} p<0.001")

## 7. Geographic Visualization

Map median response time by community area to see spatial patterns. This reveals whether slow-response areas cluster geographically and correlate with income.

In [ ]:
# Median response time and income by community area
ca_stats = df_inc.groupby("community_area").agg(
    median_response_days=("response_days", "median"),
    mean_response_days=("response_days", "mean"),
    request_count=("sr_number", "count"),
    per_capita_income=("per_capita_income", "first"),
).reset_index()

# Scatter plot: income vs response time by community area
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    ca_stats["per_capita_income"],
    ca_stats["median_response_days"],
    s=ca_stats["request_count"] / ca_stats["request_count"].max() * 300,
    c=ca_stats["median_response_days"],
    cmap="RdYlGn_r",
    alpha=0.7,
    edgecolors="white",
    linewidth=0.5,
)

# Trend line
z = np.polyfit(ca_stats["per_capita_income"].dropna(), ca_stats["median_response_days"].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(ca_stats["per_capita_income"].min(), ca_stats["per_capita_income"].max(), 100)
ax.plot(x_line, p(x_line), "--", color="#e74c3c", linewidth=2, alpha=0.7)

# Correlation
r, p_val = stats.pearsonr(ca_stats["per_capita_income"].dropna(), ca_stats["median_response_days"].dropna())
ax.set_xlabel("Per Capita Income ($)")
ax.set_ylabel("Median Response Time (Days)")
ax.set_title(f"Income vs Response Time by Community Area (r={r:.3f}, p={p_val:.3f})")
plt.colorbar(scatter, label="Median Days")
plt.tight_layout()
plt.savefig(FIG_DIR / "income_vs_response_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Interactive choropleth map of response times
import folium
import json
import requests as req

# Download Chicago community area boundaries
geo_url = "https://data.cityofchicago.org/api/geospatial/cauq-8yn6?method=export&type=GeoJSON"
geo_resp = req.get(geo_url, timeout=30)
ca_geo = geo_resp.json()

# Map community area number in geojson to our stats
for feature in ca_geo["features"]:
    props = feature["properties"]
    ca_num = int(props.get("area_numbe", props.get("area_num_1", 0)))
    row = ca_stats[ca_stats["community_area"] == ca_num]
    if not row.empty:
        props["median_response_days"] = round(float(row["median_response_days"].iloc[0]), 1)
        props["per_capita_income"] = int(row["per_capita_income"].iloc[0])
        props["request_count"] = int(row["request_count"].iloc[0])
    else:
        props["median_response_days"] = None

# Build Folium map
m = folium.Map(location=[41.8781, -87.6298], zoom_start=10, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=ca_geo,
    data=ca_stats,
    columns=["community_area", "median_response_days"],
    key_on="feature.properties.area_numbe",
    fill_color="RdYlGn_r",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Median Response Time (Days)",
    nan_fill_color="white",
).add_to(m)

# Add tooltips
folium.GeoJson(
    ca_geo,
    style_function=lambda x: {"fillOpacity": 0, "color": "gray", "weight": 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["community", "median_response_days", "per_capita_income", "request_count"],
        aliases=["Community Area", "Median Response (days)", "Per Capita Income ($)", "Total Requests"],
    ),
).add_to(m)

m.save(str(FIG_DIR / "chicago_311_map.html"))
print("Map saved to figures/chicago_311_map.html")
m

## 8. Temporal Patterns

Do equity gaps change over time? Are things getting better or worse?

In [ ]:
# Monthly median response time by income quartile
df_inc["month"] = df_inc["created_date"].dt.to_period("M")

monthly = (
    df_inc.groupby(["month", "income_quartile"])["response_days"]
    .median()
    .reset_index()
)
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
for q, color in zip(q_labels, colors):
    subset = monthly[monthly["income_quartile"] == q]
    ax.plot(subset["month_dt"], subset["response_days"], label=q, color=color, linewidth=2)

ax.set_xlabel("Month")
ax.set_ylabel("Median Response Time (Days)")
ax.set_title("Response Time Trend by Income Quartile")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "trend_by_income.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. NLP: Topic Patterns in Service Requests

We use the service request type as a proxy for complaint "topics" and analyze how request vocabulary differs across income levels. We also look at the `origin` field (phone, internet, app) — do lower-income areas rely more on phone calls while wealthier areas use the web?

In [ ]:
# Digital divide: how do people submit requests?
origin_by_q = (
    df_inc.groupby(["income_quartile", "origin"])
    .size()
    .unstack(fill_value=0)
)
origin_pct = origin_by_q.div(origin_by_q.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
origin_pct.plot.bar(stacked=True, ax=ax, colormap="Set2", edgecolor="white")
ax.set_ylabel("% of Requests")
ax.set_xlabel("")
ax.set_title("How Residents Submit 311 Requests by Income Quartile")
ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "origin_by_income.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSubmission channel breakdown:")
print(origin_pct.round(1))

In [ ]:
# TF-IDF on service request types to find distinguishing request categories per quartile
from sklearn.feature_extraction.text import TfidfVectorizer

# Build a "document" per quartile: concatenate all request type names
quartile_docs = (
    df_inc.groupby("income_quartile")["sr_type"]
    .apply(lambda x: " ".join(x.str.lower().str.replace("[^a-z ]", "", regex=True)))
)

tfidf = TfidfVectorizer(max_features=50, ngram_range=(1, 2), stop_words="english")
tfidf_matrix = tfidf.fit_transform(quartile_docs)
feature_names = tfidf.get_feature_names_out()

# Top distinguishing terms per quartile
print("Most distinctive request terms by income quartile:\n")
for i, q in enumerate(quartile_docs.index):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[-8:][::-1]
    terms = [(feature_names[j], scores[j]) for j in top_idx]
    print(f"  {q}:")
    for term, score in terms:
        print(f"    {term:<30} {score:.3f}")
    print()

## 10. Conclusions

### Summary of Findings

*(To be filled after running the analysis — the narrative will depend on what the data shows)*

**Methodology:**
- Analyzed 3 years of completed 311 service requests from the Chicago Data Portal
- Joined with Census ACS socioeconomic indicators by community area
- Divided 77 community areas into income quartiles
- Compared response times using non-parametric tests (Kruskal-Wallis, Mann-Whitney U)
- Controlled for request type to isolate the income effect
- Used bootstrap confidence intervals to estimate effect size
- Examined geographic patterns and submission channel disparities

**Limitations:**
- Response time measures time from creation to closure, which may not capture actual service delivery
- Community area income is a proxy — individual-level income data is not available
- Some request types may have standardized response workflows that mask real differences
- Duplicate requests (filtered) could bias results if duplication rates vary by area
- Socioeconomic data is a point-in-time snapshot; neighborhood income may have shifted

**Tools:** Python, Pandas, Scikit-learn, SciPy, Plotly, Folium, Socrata API

---

*Analysis by [Peter Keel](https://keelp2.github.io) · Data from [Chicago Data Portal](https://data.cityofchicago.org/) and U.S. Census ACS*

# Chicago 311: Do Wealthier Neighborhoods Get Faster City Services?\n\n**An equity analysis of 3 years of Chicago 311 service requests**\n\nChicago receives millions of 311 service requests each year — potholes, streetlights, rodent complaints, graffiti, and more. This analysis investigates whether response times and service quality vary systematically by neighborhood income level.\n\n**Research Question:** Do wealthier Chicago neighborhoods receive faster 311 service responses than lower-income neighborhoods?\n\n**Approach:**\n1. Pull 3 years of completed 311 requests with location data (14M+ records)\n2. Join with Census socioeconomic data by community area\n3. Compare response times across income quartiles\n4. Use NLP to understand what different neighborhoods report\n5. Test significance with statistical methods\n\n---